# Memory in LangChain Agents

This reference notebook compares two agent configurations:

- **No memory:** each invocation receives only the message supplied in that call.
- **Checkpointed memory:** an `InMemorySaver` stores the conversation state for a thread, allowing later invocations with the same `thread_id` to use earlier messages.

## Learning goals

- See why an agent does not automatically remember previous calls.
- Configure an in-memory checkpointer for a development example.
- Use a stable `thread_id` when invoking a checkpointed agent.
- Understand the difference between short-term thread memory and durable production storage.

## Before you run the notebook

1. Load a model-provider API key through the project's `.env` file or environment.
2. Run the cells from top to bottom.
3. The `InMemorySaver` example stores data only while the current Python process is running; it is intended for learning and testing, not durable production memory.

In [ ]:
# Load API keys and other local settings from the project's .env file.
from dotenv import load_dotenv

load_dotenv()

True

## 1. No Memory

This section intentionally creates a stateless agent. Each call below sends one new message, so the second call does not include the first call's conversation history.

In [2]:
from langchain.agents import create_agent

# Without a checkpointer, the agent has no persisted conversation state.
agent = create_agent(model="gpt-5-nano")

In [ ]:
from langchain.messages import HumanMessage

# This message contains the user's name and favorite color for this call only.
question = HumanMessage(
    content="Hello, my name is Michael and my favorite color is blue."
)

# The messages list is the complete conversation supplied to this invocation.
response = agent.invoke({"messages": [question]})

In [ ]:
from pprint import pprint

# Inspect the full state returned by the agent, including its message history.
pprint(response)

{'messages': [HumanMessage(content='Hello, My name is Michael and my favorite color is blue.', additional_kwargs={}, response_metadata={}, id='3578fca7-7bf7-4cb6-8051-8889d81323bf'),
              AIMessage(content='Nice to meet you, Michael. Blue is a great color—calming and versatile. What would you like to do today? I can chat about color ideas, suggest blue shades for outfits or decor, or help with something else you have in mind.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 508, 'prompt_tokens': 19, 'total_tokens': 527, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPZhJcNtxMTe8Qt68jjkuoPQuyJOg', 'service_tier': 'default', 'finish_reaso

In [ ]:
# This is a new invocation with no prior messages included.
question = HumanMessage(content="What is my favorite color?")

response = agent.invoke({"messages": [question]})

# The model may say it does not know because the earlier message was not supplied.
pprint(response)

{'messages': [HumanMessage(content='What is my favorite color?', additional_kwargs={}, response_metadata={}, id='5331343d-188c-4e8e-9555-4a935000bea8'),
              AIMessage(content='I don’t know your favorite color unless you tell me. Want me to guess? I can start with blue and you tell me if that’s it (yes/no). If not, I can try green, red, purple, etc.\n\nOr we can do a quick 4-question mini-quiz to determine it—your choice. Which would you prefer?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1554, 'prompt_tokens': 12, 'total_tokens': 1566, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1472, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPZonX9dKmz1bdoxUiRE82FbSv6Sn', 'service_tier': '

### No-memory summary

A model can only use the messages provided for the current invocation. Creating an agent without a checkpointer does not create durable conversation memory, so the second question cannot reliably use the first statement.

## 2. Short-Term Thread Memory

A checkpointer saves the agent state between invocations. The state is associated with a `thread_id`, so every call that should share memory must use the same identifier.

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver keeps checkpointed state in process memory.
# Use a durable checkpointer when state must survive restarts or be shared.
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(
    content="Hello, my name is Michael and my favorite color is blue."
)

# All calls in this conversation must use the same thread_id.
config = {"configurable": {"thread_id": "memory-demo-1"}}

# The checkpointer stores this message under the selected thread.
response = agent.invoke({"messages": [question]}, config)

In [ ]:
# Inspect the checkpointed state after the first turn.
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Michael and my favorite color is blue.', additional_kwargs={}, response_metadata={}, id='71923a17-d35d-4c23-82eb-af862667dc19'),
              AIMessage(content='Hi Michael! Nice to meet you. Blue is a great color. What would you like to do today? We can chat about your hobbies, pick a blue color palette, practice some English conversation, or I can share a fun fact about the color blue.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 381, 'prompt_tokens': 18, 'total_tokens': 399, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPaHGvReAL6VFG3JOBVdtfkZDB284', 'service_tier': 'default', 'finish_

In [ ]:
question = HumanMessage(content="What's my favorite color?")

# Reusing the same config lets the agent retrieve the earlier turn.
response = agent.invoke({"messages": [question]}, config)

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Michael and my favorite color is blue.', additional_kwargs={}, response_metadata={}, id='71923a17-d35d-4c23-82eb-af862667dc19'),
              AIMessage(content='Hi Michael! Nice to meet you. Blue is a great color. What would you like to do today? We can chat about your hobbies, pick a blue color palette, practice some English conversation, or I can share a fun fact about the color blue.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 381, 'prompt_tokens': 18, 'total_tokens': 399, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPaHGvReAL6VFG3JOBVdtfkZDB284', 'service_tier': 'default', 'finish_

### Memory section summary

`InMemorySaver` enables short-term memory by saving state for each thread. The second invocation can answer the color question because it uses the same `thread_id` and therefore retrieves the first turn's messages.

## Conclusion and reference checklist

The key distinction is where conversation state comes from:

- Without a checkpointer, include the complete conversation history in each request yourself.
- With a checkpointer, pass the same `thread_id` so LangGraph can restore the saved state.
- Use `InMemorySaver` for local experiments and tests. Choose a durable checkpointer for applications that need state after a process restart.
- Treat a thread as a conversation boundary: use a new identifier when a user starts a separate conversation.

The reusable invocation pattern is `agent.invoke({"messages": [message]}, config)` for checkpointed conversations, and `agent.invoke({"messages": [message]})` for stateless calls.